# Lemon のモジュールを利用して、BERT-miniモデルを作成する

BERT mini model は 以下の設定
* epoch 5 までのモデル
* batch size 32,
* linearly decreasing learning rate from 3 ^ 10−5 with 50 warmup steps
* 16-bit precision optimization
* 1, 3, 5, 10, or 20 epochs depending on the dataset size
* The final model is the one from the epoch with the highest F1 score on the validation dataset.

In [ ]:
dataset_out_root_dir = "../../data/lemon/datasets"
model_out_root_dir = "../../data/lemon/model/bert-mini"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]
gpu_id = 5


In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_id}"


In [ ]:
import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())


In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


In [ ]:
import pickle
import pathlib

import lemon.utils.datasets.deepmatcher
from transformers import AutoModelForSequenceClassification
from transformers.trainer_callback import TrainerState


def get_best_model_checkpoint_dir(checkpoints_dir_path: pathlib.Path):
    last_checkpoints_dir = sorted(
        checkpoints_dir_path.iterdir(), key=lambda x: int(str(x).split("-")[-1])
    )[-1]
    state = TrainerState.load_from_json(
        f"{str(last_checkpoints_dir)}/trainer_state.json"
    )
    return pathlib.Path((state.best_model_checkpoint))


for dataset_name in dataset_names:
    print("=============================")
    print(dataset_name)
    print("=============================")
    load_dataset_func = getattr(lemon.utils.datasets.deepmatcher, dataset_name)
    dataset = load_dataset_func(dataset_out_root_dir)
    output_dir_path = (
        pathlib.Path(model_out_root_dir) / dataset_name
    )
    output_dir_path.mkdir(parents=True, exist_ok=True)

    matcher = lemon.utils.matchers.TransformerMatcher(
        "google/bert_uncased_L-4_H-256_A-4",
        tokenizer_args={"model_max_length": 256},
        training_args={
            "output_dir": str(output_dir_path / "checkpoints"),
            "logging_dir": str(output_dir_path / "logs"),
            "per_device_train_batch_size": 32,
            "learning_rate": 3e-5,
            "warmup_steps": 50,
            "fp16": True,
            "num_train_epochs": 20,
        },
    )
    print("training...")
    ret = matcher.fit(
        dataset.train.records.a,
        dataset.train.records.b,
        dataset.train.record_id_pairs,
        dataset.train.labels,
        dataset.val.record_id_pairs,
        dataset.val.labels,
    )
    display(ret)
    print("training...done")
    eval_result = matcher.evaluate(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    print(eval_result)
    with (output_dir_path / "eval_result.pickle").open(
        "wb"
    ) as f:
        pickle.dump(eval_result, f)
    print("reload test...")
    print(get_best_model_checkpoint_dir(output_dir_path / "checkpoints"))
    bert_mini_model_reload = AutoModelForSequenceClassification.from_pretrained(
        get_best_model_checkpoint_dir(output_dir_path / "checkpoints")
    )
    matcher_reload = lemon.utils.matchers.TransformerMatcher(bert_mini_model_reload)
    eval_result_reload = matcher_reload.evaluate(
        dataset.test.records.a,
        dataset.test.records.b,
        dataset.test.record_id_pairs,
        dataset.test.labels,
    )
    print(eval_result_reload)
    assert eval_result == eval_result_reload
